# Interactions between different modules

Fits, for each of the 1,041 response genes,

```
y ~ K_0 + ... + K_5 + K_0*K_1 + ... + K_4*K_5
```

over cells carrying knockouts from two different modules together with the
single-knockout cells. The fifteen `K_i:K_j` coefficients ask whether two
modules together move a gene by more or less than the sum of their separate
effects.

Writes `ComboEffects_lm_residuals_withInteractions.rds`, read by Figures 5A,
5ACD, 5F and 5G.

## Setup

In [ ]:
%load_ext rpy2.ipython

import scanpy as sc
import pandas as pd
import anndata2ri
from rpy2.robjects import numpy2ri, pandas2ri

# The %%R cell below receives pandas frames; these converters are what the
# libraries.py star-import used to activate.
numpy2ri.activate()
pandas2ri.activate()
anndata2ri.activate()

DATASET  = "/home/eraslab1/Projects/E3Ligase/analysisSingle/Notebooks/CombinatorialPerturbations/dataset"
OUT_FILE = "outputs/ComboEffects_lm_residuals_withInteractions.rds"

MODULES = ["K_0", "K_1", "K_2", "K_3", "K_4", "K_5"]

## Response and design

The response is the `ClusterResiduals` layer — expression with the
transcriptional cluster already regressed out — so the coefficients are not
confounded by which cluster a cell belongs to.

In [ ]:
adataSingles = sc.read(f"{DATASET}/adataTrainSingles.h5ad")
adataDoubles = sc.read(f"{DATASET}/adataDoubles.h5ad")
adata = sc.AnnData.concatenate(adataSingles, adataDoubles)

expressionMatrix = pd.DataFrame(adata.layers["ClusterResiduals"])
expressionMatrix.columns = adata.var_names
expressionMatrix.index = adata.obs.index

guideMatrix = adata.obs[MODULES]
allResp = adata.var_names

# The fifteen unordered module pairs, in the order the original listed them.
interactions = [f"{a}*{b}" for i, a in enumerate(MODULES) for b in MODULES[i + 1:]]
my_formula = "y~" + "+".join(MODULES + interactions)

print(adata.shape)
print(my_formula)

## One model per gene

:::{note}
The original looped `seq(1, 1042, 1)` over 1,041 genes, so the last iteration
always failed and was swallowed by `tryCatch`. Looping over the actual column
count drops that silent error and leaves the result unchanged.
:::

In [ ]:
%%R -i guideMatrix,expressionMatrix,my_formula,allResp,OUT_FILE
library(broom)

coefDF <- data.frame()

for (i in seq_len(ncol(expressionMatrix))) {
    tryCatch({
        guideMatrix["y"] <- expressionMatrix[, i]
        myFit <- lm(formula(my_formula), data = guideMatrix)
        myDF  <- data.frame(tidy(myFit))
        myDF$respGene <- allResp[i]
        coefDF <- rbind(coefDF, myDF)
    }, error = function(e) message("gene ", i, " skipped: ", conditionMessage(e)))
}

saveRDS(coefDF, OUT_FILE)
dim(coefDF)